# ToxValDB first analysis:
This notebook serves as the initial data preparation step for the ToxValDB dataset.
It reads the unified dataset, maps species and routes to expected OECD guidelines,
and performs a first pass of text normalization on toxicological effects.

### Import packages

In [1]:
import pandas as pd
import re

### Read data:

In [2]:
df = pd.read_csv('../../00.dowload_data/outputs/ToxValDB_v9.7.0_Unified.tsv', sep='\t', low_memory=False)
print(f"Dataset shape is {df.shape}")

Dataset shape is (249071, 57)


### Create OECD expected guideline

In [3]:
def get_guideline_matrix_v3(row):
    # --- 0. PRE-PROCESSING ---
    st_type = str(row.get('STUDY_TYPE', '')).lower()
    route = str(row.get('EXPOSURE_ROUTE', '')).lower()
    species = str(row.get('SPECIES_COMMON', '')).lower()
    duration_val = str(row.get('STUDY_DURATION_VALUE', ''))
    duration_class = str(row.get('STUDY_DURATION_CLASS', '')).lower()
    generation = str(row.get('GENERATION', ''))
    toxval_type = str(row.get('TOXVAL_TYPE', '')).lower()
    year = row.get('YEAR', 0)

    # Intentional exclusion
    if 'human' in species: return None

    # --- 1. GROUP DEFINITION ---
    is_rabbit = 'rabbit' in species
    is_guinea = 'guinea' in species
    is_mouse  = 'mouse' in species
    is_rat    = 'rat' in species
    is_dog    = 'dog' in species
    is_monkey = 'monkey' in species or 'macaque' in species or 'primate' in species
    is_fish   = any(x in species for x in ['danio', 'fish', 'trout', 'carp', 'minnow'])
    is_bird   = any(x in species for x in ['quail', 'chicken', 'duck', 'mallard', 'bird'])

    # --- 2. ASSIGNMENT LOGIC ---

    # === BLOCK A: ECOTOXICOLOGY ===
    if is_fish: return "OECD 203"
    if is_bird: return "OECD 205"

    # === BLOCK B: ENDOCRINE ===
    if 'uterotrophic' in st_type: return "OECD 440"
    if 'hershberger' in st_type: return "OECD 441"

    # === BLOCK C: LOCAL EFFECTS ===
    if is_guinea: return "OECD 406" # Sensitization
    if is_rabbit and ('skin' in st_type or 'dermal' in route or 'irrit' in st_type): return "OECD 404"
    if is_rabbit and 'eye' in st_type: return "OECD 405"

    # === BLOCK D: ACUTE TOXICITY ===
    if 'acute' in st_type or 'ld50' in toxval_type:
        # Oral (Rat/Mouse)
        if 'oral' in route and (is_rat or is_mouse):
            try:
                # Try parsing year to distinguish 401 vs 423
                y_val = float(str(year).replace(',', '').split('.')[0])
                if y_val >= 2002: return "OECD 423"
                else: return "OECD 401"
            except: return "OECD 401"
        
        # Dermal (Rat/Rabbit)
        if 'dermal' in route and (is_rat or is_rabbit): return "OECD 402"
        # Inhalation (Rat)
        if 'inhal' in route and is_rat: return "OECD 403"

    # === BLOCK E: REPRO / DEVELOPMENT ===
    # Multigenerational (Rat/Mouse)
    if ('F1' in generation or 'F2' in generation or 'P0' in generation) and (is_rat or is_mouse):
        return "OECD 416"
    # Development (Rat/Rabbit/Mouse)
    if ('developmental' in st_type or 'fetus' in generation) and (is_rat or is_rabbit or is_mouse):
        return "OECD 414"
    # Reproductive Screening
    if 'reproduction' in st_type and (is_rat or is_mouse):
         if 'one' in st_type: return "OECD 415"
         return "OECD 421"

    # === BLOCK F: REPEATED DOSE (Subchronic) ===
    if 'subchronic' in st_type or 'repeat' in st_type or 'short-term' in st_type:
        # F1. Oral - Rodents
        if 'oral' in route and (is_rat or is_mouse):
            if '28' in duration_val or 'short' in duration_class: return "OECD 407"
            if '90' in duration_val or 'subchronic' in duration_class: return "OECD 408"
            return "OECD 408" # Default
        
        # F2. Oral - Non-Rodents
        if 'oral' in route and (is_dog or is_monkey):
            return "OECD 409" # 90-Day Non-Rodent

        # F3. Dermal / Inhalation
        if 'dermal' in route and (is_rat or is_rabbit or is_guinea): return "OECD 410"
        if 'inhal' in route and is_rat: return "OECD 412"

    # === BLOCK G: CHRONIC ===
    # Applies to rodents and non-rodents
    if 'chronic' in st_type and (is_rat or is_mouse or is_dog or is_monkey):
        return "OECD 452"

    return None

# Apply to DataFrame
df['MATRIX_GUIDELINE'] = df.apply(get_guideline_matrix_v3, axis=1)

### Clean dataset:

In [4]:
# Delete columns that have '-' in TOXICOLOGICAL_EFFECT
#print(f"Rows before deleting '-': {df.shape[0]}")
#df = df[~df["TOXICOLOGICAL_EFFECT"].str.contains("-", na=False)]
#print(f"Rows after deleting '-': {df.shape[0]}")

df = df.drop(columns=[col for col in df.columns if "_ORIGINAL" in col])

df = df.drop(columns=[
    'QC_CATEGORY',
    'QC_STATUS',
    'SOURCE_HASH',
    'SOURCE_URL',
    'SUBSOURCE_URL',
    'source_filename'
])

### Clean Tox. effect column

In [5]:
# --- 1. Controlled SPLIT and EXPLODE ---
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].fillna('').astype(str)
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.replace(r';', '|', regex=True)

# Deduplicate elements in the list BEFORE exploding
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.split('|').apply(lambda x: list(dict.fromkeys([i.strip() for i in x if i.strip()])) if isinstance(x, list) else x)
df = df.explode('TOXICOLOGICAL_EFFECT').reset_index(drop=True)
print(f"Rows after controlled explode: {df.shape[0]}")

# --- 2. NOISE CLEANING AND NORMALIZATION ---
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.lower().str.strip()

# A. Percentages
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.replace(r'[~><=+\/-]*\s?\d+[\d\.,]*\s?%', '', regex=True)
# B. Doses and Units
units = r'(mg\/kg(?:\/day)?|ppm|mg\/m3|mg\/l|g\/kg|µg\/kg|mmol\/l|mg\/dl|mg\/kg\s?bw)'
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.replace(r'(?:at|@)?\s?[~><=]*\s?\d+[\d\.,-]*\s?' + units + r'\b', '', regex=True)
# C. Technical parentheses
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.replace(r'\([^)]*[\d%][^)]*\)', '', regex=True)
# D. General cleanup of characters and spaces
df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Semantic noise filtering
STOP_PHRASES = {
    #"no abnormal phenotype detected", "no significant changes", "no treatment-related effects",
    #"no adverse effects", "no effects", "no observation", "no changes", "normal",
    #"unremarkable", "negative", "none", "na", "no abnormal findings",
    "terminal sacrifice", "scheduled sacrifice", "study termination", "necropsy", "autopsy"
}
df = df[~df['TOXICOLOGICAL_EFFECT'].isin(STOP_PHRASES)]

#NEGATIVE_PATTERN = r'^no\s+(?:significant|abnormal|treatment|adverse|observed|toxic|pathological|compound)'
#df = df[~df['TOXICOLOGICAL_EFFECT'].str.contains(NEGATIVE_PATTERN, case=False, na=False, regex=True)]
#df = df[df['TOXICOLOGICAL_EFFECT'].str.len() >= 4]

# Deduplication after noise filtering
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows after noise filtering: {df.shape[0]}")

# --- 3. SEX/SPECIES CLEANUP AND FINAL DEDUPLICATION ---
def remove_in_sex_species(text):
    pattern_in = r'\b(?:in|/)\s*(?:male?s?|female?s?|mice|rats?|rabbits?|dogs?|pigs?|monkeys?)(?:\s+(?:and|or)\s+(?:in\s+|/)?(?:male?s?|female?s?|mice|rats?|rabbits?|dogs?|pigs?|monkeys?))*'
    pattern_parens = r'\(\s*(?:male?s?|female?s?)(?:\s+(?:and|or)\s+(?:male?s?|female?s?))?(?:\s+only)?\s*\)'
    combined = f'({pattern_in}|{pattern_parens})'
    cleaned = re.sub(combined, '', text, flags=re.IGNORECASE | re.VERBOSE)
    return re.sub(r'\s+', ' ', cleaned).strip(' ,')

df['TOXICOLOGICAL_EFFECT'] = df['TOXICOLOGICAL_EFFECT'].apply(remove_in_sex_species)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Final clean rows: {df.shape[0]}")

Rows after controlled explode: 484941
Rows after noise filtering: 479533
Final clean rows: 479533


In [6]:
df.to_csv('../outputs/ToxValDB_v9.7.0_Cleaned_00_negatives.tsv', sep='\t', index=False)

In [7]:
df = pd.read_csv('../outputs/ToxValDB_v9.7.0_Cleaned_00_negatives.tsv', sep='\t', low_memory=False)